# **任务18 残差网络 - 卷积神经网络 | Residual Network (ResNet) - CNN**

残差网络是对传统深度神经网络的改良，而非是一种新的层结构，我们可以在任何神经网络层中应用它。

## 1. 定义残差快

由于本层输入如要与本层输出加和构建残差链接，所以要确保特征图的宽高不会变化。

In [11]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=None):
        super().__init__()
        if not hidden_dim:
            hidden_dim = input_dim

        self.layer = nn.Sequential(
            nn.Conv2d(input_dim, hidden_dim, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(),
        )
        self.output_layer = nn.Conv2d(hidden_dim, output_dim, kernel_size=3, padding=1, stride=1)

    def forward(self, x):
        out = self.layer(x)
        out = out + x       # 残差连接
        out = self.output_layer(out)
        return out

## 2. **定义模型**

使用卷积残差快直接替换CNN的卷积层即可。

In [12]:
class MainModel(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim):
        super().__init__()
        self.layer = nn.Sequential(
            ResBlock(input_dim, hidden_dim),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            ResBlock(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            ResBlock(hidden_dim, hidden_dim),
            nn.MaxPool2d(2, 2),
            nn.Flatten()
        )
        self.output_layer = nn.Linear(1024, output_dim)

    def forward(self, x):
        out = self.layer(x)
        out = self.output_layer(out)
        return out

## 3. **模拟前向传播**

In [13]:
batch_size = 4
input_tensor = torch.rand((batch_size, 3, 64, 64))

model = MainModel(3, 5, 16)

# 前向传播过程
output_tensor = model(input_tensor)

print('input_tensor:', input_tensor.shape)
print('output_tensor:', output_tensor.shape)


input_tensor: torch.Size([4, 3, 64, 64])
output_tensor: torch.Size([4, 5])


## 4. **总结**

残差连接是残差网络的关键，不止线性层和卷积层可以使用残差连接，实际上只要是本层输入与本层输出形状相同的位置，都可以使用残差连接，比如一些注意力机制，都可以尝试使用残差连接增强其性能。